In [1]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

print(v1.dot(dv))
print(v2.dot(dv))

2026-06-29 22:31:44.091984248 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


0.3233238799303238
0.019730422395141473


In [2]:
from ingest import load_faq_data
from tqdm.auto import tqdm
import numpy as np

documents = load_faq_data()

texts = [
    doc["question"] + " " + doc["answer"]
    for doc in documents
]

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    vectors = embed.encode_batch(batch)
    X.extend(vectors)

X = np.array(X)

X.shape

  0%|          | 0/27 [00:00<?, ?it/s]

(1350, 384)

In [3]:
query = "Can I still join the course after the start date?"

v_query = embed.encode(query)

scores = X.dot(v_query)

idx = scores.argmax()

print(scores[idx])
documents[idx]

0.7629411421659094


{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [4]:
top5 = np.argsort(-scores)[:5]

for idx in top5:
    print(scores[idx])
    print(documents[idx]["course"])
    print(documents[idx]["question"])
    print(documents[idx]["answer"])
    print()

0.7629411421659094
data-engineering-zoomcamp
Course: Can I still join the course after the start date?
Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.

0.7579371360784597
mlops-zoomcamp
Course - Can I still join the course after the start date?
Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.

Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.

0.7192131018586465
machine-learning-zoomcamp
The course has already started. Can I still join it?
Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.

In order to get a certificate